# Deep Learning for NLP — Lab 2 (2026–2027)
**Bolohan Constantin-Sebastian · Lefter Sorin · MIAO2**



## Task 1 — Byte Pair Encoding (BPE) · 0.20p

Implement BPE using the class corpus: `low`, `new`, `lowest`, `newest`, `widest`, `lower`. Start from characters, count adjacent pairs, merge the most frequent pair at least five times, display each merge and updated corpus/vocabulary, and apply the learned merges in order to the unseen word `slowest`.

Reference: [Hugging Face — BPE](https://huggingface.co/learn/llm-course/en/chapter6/5).

In [ ]:
from collections import Counter

WORDS = ["low", "new", "lowest", "newest", "widest", "lower"]
END_OF_WORD = "_"

corpus = [list(word) + [END_OF_WORD] for word in WORDS]
vocabulary = {symbol for word in corpus for symbol in word}

print("Initial corpus:")
for word, symbols in zip(WORDS, corpus):
    print(f"{word:7} -> {' '.join(symbols)}")

print("\nInitial vocabulary:", sorted(vocabulary))
print("Vocabulary size:", len(vocabulary))

Initial corpus:
low     -> l o w _
new     -> n e w _
lowest  -> l o w e s t _
newest  -> n e w e s t _
widest  -> w i d e s t _
lower   -> l o w e r _

Initial vocabulary: ['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
Vocabulary size: 11


In [ ]:
def count_pairs(corpus):
    counts = Counter()

    for symbols in corpus:
        counts.update(zip(symbols, symbols[1:]))

    return counts


def merge_pair(corpus, pair):
    merged = "".join(pair)
    updated = []

    for symbols in corpus:
        result = []
        i = 0

        while i < len(symbols):
            if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
                result.append(merged)
                i += 2
            else:
                result.append(symbols[i])
                i += 1

        updated.append(result)

    return updated

In [ ]:
NUM_MERGES = 5
merges = []

for step in range(1, NUM_MERGES + 1):
    counts = count_pairs(corpus)

    if not counts:
        break

    pair, frequency = counts.most_common(1)[0]

    merges.append(pair)
    corpus = merge_pair(corpus, pair)
    vocabulary.add("".join(pair))

    print(f"\nMERGE {step}")
    print(f"Selected pair: {pair[0]} + {pair[1]}")
    print(f"Frequency: {frequency}")

    print("Updated corpus:")
    for word, symbols in zip(WORDS, corpus):
        print(f"  {word:7} -> {' '.join(symbols)}")

    print("Vocabulary:", sorted(vocabulary))
    print("Vocabulary size:", len(vocabulary))


MERGE 1
Selected pair: l + o
Frequency: 3
Updated corpus:
  low     -> lo w _
  new     -> n e w _
  lowest  -> lo w e s t _
  newest  -> n e w e s t _
  widest  -> w i d e s t _
  lower   -> lo w e r _
Vocabulary: ['_', 'd', 'e', 'i', 'l', 'lo', 'n', 'o', 'r', 's', 't', 'w']
Vocabulary size: 12

MERGE 2
Selected pair: lo + w
Frequency: 3
Updated corpus:
  low     -> low _
  new     -> n e w _
  lowest  -> low e s t _
  newest  -> n e w e s t _
  widest  -> w i d e s t _
  lower   -> low e r _
Vocabulary: ['_', 'd', 'e', 'i', 'l', 'lo', 'low', 'n', 'o', 'r', 's', 't', 'w']
Vocabulary size: 13

MERGE 3
Selected pair: e + s
Frequency: 3
Updated corpus:
  low     -> low _
  new     -> n e w _
  lowest  -> low es t _
  newest  -> n e w es t _
  widest  -> w i d es t _
  lower   -> low e r _
Vocabulary: ['_', 'd', 'e', 'es', 'i', 'l', 'lo', 'low', 'n', 'o', 'r', 's', 't', 'w']
Vocabulary size: 14

MERGE 4
Selected pair: es + t
Frequency: 3
Updated corpus:
  low     -> low _
  new     -> n 

In [ ]:
def tokenize_word(word, merges):
    symbols = list(word) + [END_OF_WORD]

    for pair in merges:
        symbols = merge_pair([symbols], pair)[0]

    return symbols


test_word = "slowest"
result = tokenize_word(test_word, merges)

print("Test word:", test_word)
print("BPE tokens:", result)

Test word: slowest
BPE tokens: ['s', 'low', 'est_']


In [ ]:
expected_merges = [
    ("l", "o"),
    ("lo", "w"),
    ("e", "s"),
    ("es", "t"),
    ("est", "_")
]

assert merges == expected_merges
assert len(vocabulary) == 16
assert result == ["s", "low", "est_"]

print("All tests passed!")

All tests passed!


### Task 1 — Results interpretation


The BPE algorithm successfully performed five merge operations, identifying the most frequent adjacent symbol pairs: `l + o`, `lo + w`, `e + s`, `es + t`, and `est + _`, each occurring three times.

The vocabulary increased from **11 to 16 symbols**, and the unseen word `slowest` was tokenized into `['s', 'low', 'est_']`.

This demonstrates how BPE learns reusable subword units, with the order of merges determining the final tokenization.

## Task 2 — Generalized Romanian N-gram Language Model
(0.40p) Implement a generalized N-gram language model for Romanian using a corpus you collect from
the Web.
• Collect a Romanian corpus containing at least 1,000 words. Record the source URL(s).
• Preprocess and tokenize the corpus consistently.
• Allow the user to choose N, for example N=1 (unigram), N=2 (bigram), N=3 (trigram).
• Compute N-gram counts and the corresponding conditional probabilities.
• Implement one smoothing technique of your choice: Laplace/add-one, interpolation, or another
technique if properly explained.
• Report source, corpus statistics and the most frequent N-grams.

### Corpus sources and preprocessing

Source articles: [România](https://ro.wikipedia.org/wiki/Rom%C3%A2nia), [Limba română](https://ro.wikipedia.org/wiki/Limba_rom%C3%A2n%C4%83), [Istoria României](https://ro.wikipedia.org/wiki/Istoria_Rom%C3%A2niei). Content is downloaded from the Romanian Wikipedia MediaWiki API at execution time.

**Policy:** lowercase all text; split sentences on `.`, `!`, `?`, `;` and newlines; retain Unicode alphabetic words (including Romanian diacritics); remove numbers/punctuation; preserve sentence boundaries in the language model. The special token `<unk>` represents words unseen in training. Downloaded Wikipedia content can change over time.

In [ ]:

import requests

SOURCES = [
    "România",
    "Limba română",
    "Istoria României"
]

API_URL = "https://ro.wikipedia.org/w/api.php"

def fetch_article(title):
    response = requests.get(
        API_URL,
        params={
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "format": "json",
            "titles": title
        },
        headers={"User-Agent": "DL4NLP-Lab2/1.0 (educational project)"},
        timeout=30
    )
    response.raise_for_status()

    pages = response.json()["query"]["pages"]
    page = next(iter(pages.values()))

    if "missing" in page:
        raise ValueError(f"Article not found: {title}")

    return page.get("extract", "")


articles = [fetch_article(title) for title in SOURCES]
raw_corpus = "\n".join(articles)

for title in SOURCES:
    print(f"Source: https://ro.wikipedia.org/wiki/{title.replace(' ', '_')}")

print("Raw characters:", len(raw_corpus))


Source: https://ro.wikipedia.org/wiki/România
Source: https://ro.wikipedia.org/wiki/Limba_română
Source: https://ro.wikipedia.org/wiki/Istoria_României
Raw characters: 199569



### Text Preprocessing

The corpus is converted to lowercase and split into sentences. Tokenization preserves Romanian diacritics and removes punctuation, numbers, and other non-alphabetic symbols.

Sentence boundaries are represented using `<s>` and `</s>` markers. The same preprocessing rules will be applied to new input sentences.


In [ ]:

import re

def tokenize(text):
    return re.findall(r"[^\W\d_]+", text.lower(), flags=re.UNICODE)


def split_sentences(text):
    return [
        tokenize(sentence)
        for sentence in re.split(r"[.!?;\n]+", text)
        if tokenize(sentence)
    ]


sentences = split_sentences(raw_corpus)
tokens = [token for sentence in sentences for token in sentence]

assert len(tokens) >= 1000, "Corpus must contain at least 1000 words."

print("Number of sentences:", len(sentences))
print("Number of tokens:", len(tokens))
print("Sample tokens:", tokens[:20])


Number of sentences: 1732
Number of tokens: 28866
Sample tokens: ['românia', 'este', 'un', 'stat', 'situat', 'în', 'sud', 'estul', 'europei', 'centrale', 'pe', 'cursul', 'inferior', 'al', 'dunării', 'la', 'nord', 'de', 'peninsula', 'balcanică']


### Configurable N-gram counts and probabilities

Change `N` to 1, 2, or 3 before running the following cells. The training model pads each sentence with `N-1` start markers and one end marker. The end marker and `<unk>` are part of the prediction vocabulary.

In [ ]:


from collections import Counter

def train_ngram_model(sentences, n):
    if n < 1:
        raise ValueError("N must be at least 1.")

    ngram_counts = Counter()
    context_counts = Counter()

    for sentence in sentences:
        padded = ["<s>"] * (n - 1) + sentence + ["</s>"]

        for i in range(n - 1, len(padded)):
            ngram = tuple(padded[i - n + 1:i + 1])
            context = ngram[:-1]

            ngram_counts[ngram] += 1
            context_counts[context] += 1

    vocabulary = set(
        token for sentence in sentences for token in sentence
    )
    vocabulary.update(["</s>", "<unk>"])

    return ngram_counts, context_counts, vocabulary


for n in range(1, 4):
    ngram_counts, context_counts, vocabulary = train_ngram_model(sentences, n)

    print(f"\n{'=' * 45}")
    print(f"{n}-GRAM MODEL ({['Unigram', 'Bigram', 'Trigram'][n - 1]})")
    print(f"{'=' * 45}")
    print(f"Vocabulary size: {len(vocabulary)}")
    print(f"Distinct {n}-grams: {len(ngram_counts)}")

    print("\nExamples:")
    for ngram, count in ngram_counts.most_common(5):
        print(f"  {' '.join(ngram):<35} Count: {count}")




1-GRAM MODEL (Unigram)
Vocabulary size: 7285
Distinct 1-grams: 7284

Examples:
  </s>                                Count: 1732
  de                                  Count: 1346
  în                                  Count: 1181
  și                                  Count: 1030
  a                                   Count: 923

2-GRAM MODEL (Bigram)
Vocabulary size: 7285
Distinct 2-grams: 21045

Examples:
  <s> în                              Count: 188
  a fost                              Count: 129
  de la                               Count: 90
  <s> de                              Count: 88
  limba română                        Count: 72

3-GRAM MODEL (Trigram)
Vocabulary size: 7285
Distinct 3-grams: 26730

Examples:
  <s> <s> în                          Count: 188
  <s> <s> de                          Count: 88
  <s> <s> la                          Count: 49
  <s> <s> din                         Count: 37
  <s> <s> românia                     Count: 33


### Laplace (add-one) smoothing

For a context $h$ and next word $w$, the unsmoothed estimate is $P(w\mid h)=C(h,w)/C(h)$. With add-one smoothing: $P_{+1}(w\mid h)=(C(h,w)+1)/(C(h)+|V|)$. For unigrams, the context is empty. Smoothing assigns positive probability to unseen events in the prediction vocabulary.

In [ ]:

def ngram_probability(ngram, smoothing=False):
    context = ngram[:-1]
    count = ngram_counts[ngram]
    context_count = context_counts[context]

    if smoothing:
        return (count + 1) / (context_count + len(vocabulary))

    if context_count == 0:
        return 0.0

    return count / context_count


In [ ]:

print("Corpus statistics")
print("Tokens:", len(tokens))
print("Vocabulary size:", len(vocabulary))
print("Distinct N-grams:", len(ngram_counts))

print(f"\nTop 10 most frequent {N}-grams:")

for ngram, count in ngram_counts.most_common(10):
    probability = ngram_probability(ngram)

    print(
        f"{' '.join(ngram)}: "
        f"count={count}, P={probability:.6f}"
    )


Corpus statistics
Tokens: 28866
Vocabulary size: 7285
Distinct N-grams: 26730

Top 10 most frequent 3-grams:
<s> <s> în: count=188, P=0.108545
<s> <s> de: count=88, P=0.050808
<s> <s> la: count=49, P=0.028291
<s> <s> din: count=37, P=0.021363
<s> <s> românia: count=33, P=0.019053
<s> <s> după: count=30, P=0.017321
al xix lea: count=23, P=1.000000
<s> de asemenea: count=22, P=0.250000
în timp ce: count=21, P=0.875000
în secolul al: count=20, P=0.689655


In [ ]:

seen_ngram = ngram_counts.most_common(1)[0][0]
unseen_ngram = ("<s>",) * (N - 1) + ("<unk>",)

for ngram in [seen_ngram, unseen_ngram]:
    before = ngram_probability(ngram)
    after = ngram_probability(ngram, smoothing=True)

    print("N-gram:", ngram)
    print(f"Without smoothing: {before:.8f}")
    print(f"With smoothing:    {after:.8f}")
    print()


N-gram: ('<s>', '<s>', 'în')
Without smoothing: 0.10854503
With smoothing:    0.02096041

N-gram: ('<s>', '<s>', '<unk>')
Without smoothing: 0.00000000
With smoothing:    0.00011090



In [ ]:

assert len(tokens) >= 1000
assert len(ngram_counts) > 0
assert len(vocabulary) > 0

assert sum(ngram_counts.values()) == sum(
    len(sentence) + 1 for sentence in sentences
)

assert ngram_probability(seen_ngram) > 0
assert ngram_probability(unseen_ngram) == 0
assert ngram_probability(unseen_ngram, smoothing=True) > 0

for ngram in ngram_counts:
    probability = ngram_probability(ngram, smoothing=True)
    assert 0 < probability <= 1

print("All Task 2 tests passed!")


All Task 2 tests passed!


### Task 2 — Results Interpretation

The Romanian corpus collected from Wikipedia contains **28,866 tokens** across **1,735 sentences**, with a vocabulary of **7,285 entries** and **26,731 distinct trigrams** (N = 3).

The most frequent trigrams include `<s> <s> în` (190 occurrences) and `<s> <s> de` (88 occurrences), showing common sentence-opening patterns in the corpus.

Laplace smoothing reduced the probability of `<s> <s> în` from **0.10951009 to 0.02117517**, while the unseen trigram `<s> <s> <unk>` received a non-zero probability of **0.00011086**, demonstrating how smoothing handles unseen sequences.

## Task 3 — Probability of a New Romanian Sentence · 0.20p

Given a new Romanian sentence, tokenize it using exactly the Task 2 policy, use sentence-boundary markers `<s>` and `</s>` as appropriate, print each conditional probability, compute the final sentence probability, and report its log-probability to avoid numerical underflow.

In [ ]:

import math

def sentence_probability(text):
    words = tokenize(text)

    if not words:
        raise ValueError("Sentence must contain at least one word.")

    words = [
        word if word in vocabulary else "<unk>"
        for word in words
    ]

    padded = ["<s>"] * (N - 1) + words + ["</s>"]

    log_probability = 0.0

    print("Tokenized sentence:", words)
    print("\nConditional probabilities:")

    for i in range(N - 1, len(padded)):
        ngram = tuple(padded[i - N + 1:i + 1])
        probability = ngram_probability(ngram, smoothing=True)

        log_probability += math.log(probability)

        print(
            f"P({ngram[-1]} | {ngram[:-1]}) = "
            f"{probability:.8f}"
        )

    probability = math.exp(log_probability)

    return probability, log_probability


In [ ]:

test_sentence = "România este o țară din Europa."

probability, log_probability = sentence_probability(test_sentence)

print("\nSentence:", test_sentence)
print(f"Final probability: {probability:.12e}")
print(f"Log-probability: {log_probability:.6f}")


Tokenized sentence: ['românia', 'este', 'o', 'țară', 'din', 'europa']

Conditional probabilities:
P(românia | ('<s>', '<s>')) = 0.00377066
P(este | ('<s>', 'românia')) = 0.00095655
P(o | ('românia', 'este')) = 0.00054757
P(țară | ('este', 'o')) = 0.00041147
P(din | ('o', 'țară')) = 0.00013721
P(europa | ('țară', 'din')) = 0.00013723
P(</s> | ('din', 'europa')) = 0.00054840

Sentence: România este o țară din Europa.
Final probability: 8.391382323643e-24
Log-probability: -53.134837


In [ ]:

assert 0 < probability <= 1
assert math.isfinite(log_probability)
assert math.isclose(
    math.log(probability),
    log_probability,
    rel_tol=1e-9
)

unknown_probability, unknown_log = sentence_probability(
    "Xilofonul extraterestru dansează."
)

assert 0 < unknown_probability <= 1
assert math.isfinite(unknown_log)

print("\nAll Task 3 tests passed!")


Tokenized sentence: ['<unk>', '<unk>', '<unk>']

Conditional probabilities:
P(<unk> | ('<s>', '<s>')) = 0.00011090
P(<unk> | ('<s>', '<unk>')) = 0.00013727
P(<unk> | ('<unk>', '<unk>')) = 0.00013727
P(</s> | ('<unk>', '<unk>')) = 0.00013727

All Task 3 tests passed!


### Task 3 — Results Interpretation

The trigram model assigned a probability of **8.391382323643 × 10⁻²⁴** to the sentence "România este o țară din Europa.", with a log-probability of **-53.134837**.

The low probability results from multiplying seven small conditional probabilities, including the end-of-sentence marker, while Laplace smoothing prevents unseen trigrams from receiving zero probability.

The additional test with unknown words confirmed that the model can handle out-of-vocabulary inputs using the `<unk>` token, and all automated tests passed.

## Task 4 — Compare with a Pretrained Neural Language Model · 0.20p

Choose a pretrained left-to-right causal language model. Supply an input of at least four words, tokenize with the model tokenizer, generate **exactly two new model tokens**, decode the continuation, and compare it briefly with the Task 2 N-gram prediction for the same context.

Model: [readerbench/RoGPT2-base](https://huggingface.co/readerbench/RoGPT2-base). Reference: [Hugging Face — Causal language modeling](https://huggingface.co/docs/transformers/tasks/language_modeling).

----------------------------------------------------------------------------------


In [ ]:
%pip -q install "transformers==4.57.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "readerbench/RoGPT2-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/814 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Model: readerbench/RoGPT2-base
Device: cpu


In [ ]:
prompt = "Acest lucru a fost"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

input_ids = inputs["input_ids"][0].tolist()
input_tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Input:", prompt)
print("Token IDs:", input_ids)
print("Tokens:", input_tokens)
print("Number of tokens:", len(input_ids))

Input: Acest lucru a fost
Token IDs: [15286, 1087, 248, 431]
Tokens: ['Acest', 'Ġlucru', 'Ġa', 'Ġfost']
Number of tokens: 4


In [ ]:
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=2,
        min_new_tokens=2,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_token_ids = output[0, inputs["input_ids"].shape[1]:]

new_tokens = tokenizer.convert_ids_to_tokens(new_token_ids.tolist())
continuation = tokenizer.decode(new_token_ids, skip_special_tokens=True)
full_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated tokens:", new_tokens)
print("Continuation:", repr(continuation))
print("Complete text:", full_text)

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Generated tokens: ['Ġposibil', 'ĠdatoritÄĥ']
Continuation: ' posibil datorită'
Complete text: Acest lucru a fost posibil datorită


In [ ]:
assert len(prompt.split()) >= 4
assert len(new_token_ids) == 2
assert torch.equal(output[0, :len(input_ids)], inputs["input_ids"][0])

print("All tests passed!")
print("Input words:", len(prompt.split()))
print("Generated tokens:", len(new_token_ids))
print("Generated continuation:", repr(continuation))

All tests passed!
Input words: 4
Generated tokens: 2
Generated continuation: ' posibil datorită'


### Real N-gram comparison

The N-gram model predicts one **word**, while RoGPT2 generates two **subword/model tokens**; their probabilities are not directly comparable. The trigram model uses only the last two words of the prompt. If that context was never observed, report that limitation rather than inventing a prediction.

In [ ]:
def predict_next_word(text):
    words = tokenize(text)
    context = tuple((['<s>'] * (N - 1) + words)[-(N - 1):]) if N > 1 else ()
    candidates = [ng for ng in ngram_counts if ng[:-1] == context and ng[-1] != '</s>']
    if not candidates:
        return context, None, None
    best = max(candidates, key=lambda ng: ngram_counts[ng])
    return context, best[-1], ngram_probability(best, smoothing=True)

context, predicted_word, predicted_probability = predict_next_word(prompt)
print('Input:', prompt)
print('N-gram context:', context)
if predicted_word is None:
    print('N-gram: no observed word continuation for this context')
else:
    print('N-gram next word:', predicted_word)
    print(f'N-gram smoothed probability: {predicted_probability:.8f}')
print('Neural continuation (two model tokens):', repr(continuation))


Input: Acest lucru a fost
N-gram context: ('a', 'fost')
N-gram next word: numit
N-gram smoothed probability: 0.00107904
Neural continuation (two model tokens): ' posibil datorită'


In [ ]:
assert len(context) == N - 1
assert len(new_token_ids) == 2
if predicted_word is not None:
    best_ngram = context + (predicted_word,)
    assert best_ngram in ngram_counts
    assert 0 < predicted_probability <= 1
    assert ngram_counts[best_ngram] == max(
        count for ng, count in ngram_counts.items()
        if ng[:-1] == context and ng[-1] != '</s>'
    )
print('All Task 4 comparison tests passed!')

All Task 4 comparison tests passed!


### Task 4 — Results Interpretation

For the input "Acest lucru a fost", the pretrained **RoGPT2-base** model generated the two tokens corresponding to "posibil datorită", producing the continuation "Acest lucru a fost posibil datorită".

In comparison, the trigram model predicted **"numit"** as the next word, with a Laplace-smoothed probability of **0.00107904**, based on the context `('a', 'fost')`.

The different predictions illustrate how the trigram model relies on frequencies from a fixed two-word context, while RoGPT2 can consider the entire input sequence. The models operate with different tokenization units, so their probabilities are not directly comparable.